In [ ]:
from pathlib import Path
import os
import numpy as np
import scipy.io

In [ ]:
DATASETS = ['NC', 'OH']

RAW_RESULT_FILES = [
    f'results_{dataset}{q}.mat'
    for dataset in DATASETS
    for q in range(1, 5)
]

def find_wind_raw_data_dir():
    """Find the raw wind result files using the same style as the SDDP notebooks."""
    env_dir = os.environ.get('WIND_DATA_DIR')
    candidates = []
    if env_dir:
        candidates.append(Path(env_dir).expanduser())
    candidates.extend([
        Path.cwd() / 'data' / 'wind_data',
        Path.cwd() / 'data',
        Path.cwd() / 'wind_data',
        Path.cwd(),
        Path.cwd().parent / 'data' / 'wind_data',
        Path.cwd().parent / 'data',
        Path.cwd().parent / 'wind_data',
    ])

    for d in candidates:
        if all((d / name).exists() for name in RAW_RESULT_FILES):
            return d

    searched = '\n'.join(str(d) for d in candidates)
    raise FileNotFoundError(
        'Could not find the raw results_NC*.mat, results_OH*.mat, and results_RI*.mat files. '
        'Place them in data/wind_data, data, wind_data, the notebook working directory, '
        'or set WIND_DATA_DIR.\nSearched:\n' + searched
    )

In [ ]:
DATA_DIR = find_wind_raw_data_dir()

OUTPUT_DIR = Path(os.environ.get('WIND_OUTPUT_DIR', str(DATA_DIR))).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

T = 7
H = 24
NUM_DAYS = T + 1
REQUIRED_HOURS = NUM_DAYS * H


OVERWRITE_STANDARD_ALL_FILES = False

NC_OUT = OUTPUT_DIR / f'NC_all.mat'
OH_OUT = OUTPUT_DIR / f'OH_all.mat'
PRICE_OUT = OUTPUT_DIR / f'price_all.mat'

if OVERWRITE_STANDARD_ALL_FILES:
    NC_OUT = OUTPUT_DIR / 'NC_all.mat'
    OH_OUT = OUTPUT_DIR / 'OH_all.mat'
    PRICE_OUT = OUTPUT_DIR / 'price_all.mat'

print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('T:', T, 'NUM_DAYS:', NUM_DAYS, 'REQUIRED_HOURS:', REQUIRED_HOURS)
print('Outputs:')
print(' ', NC_OUT)
print(' ', OH_OUT)
print(' ', PRICE_OUT)

In [ ]:
def load_raw_result(dataset: str, quarter: int):
    dataset = dataset.upper()
    path = DATA_DIR / f'results_{dataset}{quarter}.mat'
    if not path.exists():
        raise FileNotFoundError(path)
    mat = scipy.io.loadmat(path)
    required = ['weekly_power', 'weekly_wind', 'weekly_price']
    missing = [key for key in required if key not in mat]
    if missing:
        raise KeyError(f'{path.name} is missing {missing}')
    return {
        'path': path,
        'weekly_power': np.asarray(mat['weekly_power'], dtype=float),
        'weekly_wind': np.asarray(mat['weekly_wind'], dtype=float),
        'weekly_price': np.asarray(mat['weekly_price'], dtype=float),
    }


def truncate_weekly(arr, required_hours=REQUIRED_HOURS, name='array'):
    arr = np.asarray(arr, dtype=float)
    if arr.ndim != 2:
        raise ValueError(f'{name} must be 2D trajectory-by-hour, got {arr.shape}')
    if arr.shape[1] < required_hours:
        raise ValueError(
            f'{name} needs at least {required_hours} hourly columns, got {arr.shape[1]}'
        )
    out = arr[:, :required_hours].copy()
    if not np.all(np.isfinite(out)):
        raise ValueError(f'{name} contains non-finite values after truncation')
    return out


def make_cell_array(items):
    cell = np.empty((1, len(items)), dtype=object)
    for idx, item in enumerate(items):
        cell[0, idx] = np.asarray(item, dtype=float)
    return cell


def split_daily_for_check(weekly):
    weekly = np.asarray(weekly, dtype=float)
    daily = np.zeros((NUM_DAYS, weekly.shape[0], H), dtype=float)
    for t in range(NUM_DAYS):
        daily[t] = weekly[:, t * H:(t + 1) * H]
    return daily

In [ ]:
nc_power, nc_wind = [], []
oh_power, oh_wind = [], []
price_cells = []

for q in range(1, 5):
    nc = load_raw_result('NC', q)
    oh = load_raw_result('OH', q)

    nc_q_power = truncate_weekly(nc['weekly_power'], name=f'NC q{q} weekly_power')
    nc_q_wind = truncate_weekly(nc['weekly_wind'], name=f'NC q{q} weekly_wind')
    nc_q_price = truncate_weekly(nc['weekly_price'], name=f'NC q{q} weekly_price')

    oh_q_power = truncate_weekly(oh['weekly_power'], name=f'OH q{q} weekly_power')
    oh_q_wind = truncate_weekly(oh['weekly_wind'], name=f'OH q{q} weekly_wind')
    oh_q_price = truncate_weekly(oh['weekly_price'], name=f'OH q{q} weekly_price')

    for label, other_price in [('OH', oh_q_price)]:
        if not np.array_equal(nc_q_price, other_price):
            max_diff = float(np.max(np.abs(nc_q_price - other_price)))
            raise ValueError(f'NC/{label} price mismatch in quarter {q}; max diff={max_diff}')


    nc_power.append(nc_q_power)
    nc_wind.append(nc_q_wind)
    oh_power.append(oh_q_power)
    oh_wind.append(oh_q_wind)
    price_cells.append(nc_q_price)

    print(
        f'q{q}: NC power {nc_q_power.shape}, OH power {oh_q_power.shape}, price {nc_q_price.shape} '
    )

In [ ]:
NC_all = {
    'weekly_power': make_cell_array(nc_power),
    'weekly_wind': make_cell_array(nc_wind),
    'T': np.asarray([[T]], dtype=np.int64),
    'NUM_DAYS': np.asarray([[NUM_DAYS]], dtype=np.int64),
    'H': np.asarray([[H]], dtype=np.int64),
}

OH_all = {
    'weekly_power': make_cell_array(oh_power),
    'weekly_wind': make_cell_array(oh_wind),
    'T': np.asarray([[T]], dtype=np.int64),
    'NUM_DAYS': np.asarray([[NUM_DAYS]], dtype=np.int64),
    'H': np.asarray([[H]], dtype=np.int64),
}

price_all = {
    'weekly_price': make_cell_array(price_cells),
    'T': np.asarray([[T]], dtype=np.int64),
    'NUM_DAYS': np.asarray([[NUM_DAYS]], dtype=np.int64),
    'H': np.asarray([[H]], dtype=np.int64),
}

scipy.io.savemat(NC_OUT, NC_all)
scipy.io.savemat(OH_OUT, OH_all)
scipy.io.savemat(PRICE_OUT, price_all)

print('Saved:')
print(' ', NC_OUT)
print(' ', OH_OUT)
print(' ', PRICE_OUT)

In [ ]:
# Diagnostics
for label, path, keys in [
    ('NC', NC_OUT, ['weekly_power', 'weekly_wind']),
    ('OH', OH_OUT, ['weekly_power', 'weekly_wind']),
    ('PRICE', PRICE_OUT, ['weekly_price']),
]:
    mat = scipy.io.loadmat(path)
    print('\n', label, path.name)
    for key in keys:
        cell = mat[key]
        print(' ', key, cell.shape, cell.dtype)
        for q in range(4):
            arr = np.asarray(cell[0, q], dtype=float)
            daily = split_daily_for_check(arr)
            print(
                f'   q{q+1}: weekly={arr.shape}, daily={daily.shape}, '
                f'min={np.nanmin(arr):.6g}, max={np.nanmax(arr):.6g}, nan={np.isnan(arr).sum()}'
            )

print('\nOriginal split reminder:')
print('  rows 0:65 are the original MATLAB training pool')
print('  rows 65:130 are the original MATLAB OOS pool')